In [6]:
import pandas as pd
import numpy as np
from syto.data.atlases.uxm_atlases import UXMMethylationAtlas
from syto.data.atlases.celfieish_atlases import CpGBetaCountsMethylationAtlas
atlas_path = "../../UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"
atlas = UXMMethylationAtlas("U25.l4.hg38", "hg38",atlas_path)
import json
from typing import Dict

In [7]:
target_regions = atlas.atlas["name"].to_list()

In [17]:
import pyarrow.dataset as ds
import pyarrow as pa

# base = "/mnt/data/loyfer/parsed_dataset/U250.l4.hg38/staged/"
base = "/mnt/data/loyfer/parsed_dataset/U25.l4.hg38/labelers_on_train_no_pattern_len_filter/final"
dset = ds.dataset(base, format="parquet")

# take the schema of one fragment and force region_bucket to a plain int
frag_schema = next(dset.get_fragments()).physical_schema
schema = frag_schema.set(
    frag_schema.get_field_index("region_bucket"),
    pa.field("region_bucket", pa.int64()),
)

dset = ds.dataset(base, format="parquet", schema=schema)
tbl = dset.to_table(filter=ds.field("name").isin(target_regions), 
                    columns =['chromosome', 'read_start', 'input_ids', 'methylation_ids', 'read_end',
       'original_label', 'name', 'region_start', 'region_end', 'dmr_ctype',
       'dmr_ctype_matched', 'dmr_ctype_label', 'file', 'hard_label_with_background', 'region_bucket', 'split'])
df = tbl.to_pandas()

In [18]:
df

,chromosome,read_start,input_ids,methylation_ids,read_end,original_label,name,region_start,region_end,dmr_ctype,dmr_ctype_matched,dmr_ctype_label,file,hard_label_with_background,region_bucket,split
0,chr1,1262135,CGTGAGCCACCGCGCCC,12222222221212222,1262151,0,chr1:1262136-1262432,1262136,1262432,Eryth-prog,Eryth-prog,15,GSM5652176_Adipocytes-Z000000T7.hg38_reads,39,0,valid
1,chr1,1262301,GGCAATGACACGATGATGGAGGGCCCACCCTACAGATGTACTCCCG...,2222222222122222222222222222222222222222222212...,1262431,30,chr1:1262136-1262432,1262136,1262432,Eryth-prog,Eryth-prog,15,GSM5652253_Pancreas-Alpha-Z00000453.hg38_reads,39,0,train
2,chr1,1262298,ATGGGCAATGACACGATGATGGAGGGCCCACCCTACAGATGTACTC...,2222222222222122222222222222222222222222222222...,1262431,30,chr1:1262136-1262432,1262136,1262432,Eryth-prog,Eryth-prog,15,GSM5652253_Pancreas-Alpha-Z00000453.hg38_reads,39,0,train
3,chr1,1262295,GTCATGGGCAATGACACGATGATGGAGGGCCCACCCTACAGATGTA...,2222222222222222122222222222222222222222222222...,1262431,30,chr1:1262136-1262432,1262136,1262432,Eryth-prog,Eryth-prog,15,GSM5652253_Pancreas-Alpha-Z00000453.hg38_reads,39,0,train
4,chr1,1262291,GCTGGTCATGGGCAATGACACGATGATGGAGGGCCCACCCTACAGA...,2222222222222222222212222222222222222222222222...,1262431,30,chr1:1262136-1262432,1262136,1262432,Eryth-prog,Eryth-prog,15,GSM5652253_Pancreas-Alpha-Z00000453.hg38_reads,39,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19241504,chr5,140679725,AACCTGAGCCAAGTGAGTGCCAATCCATCCAAAGTCTCAAGAGCCC...,2222222222222222222222222222222222222222222222...,140679820,6,chr5:140678758-140679821,140678758,140679821,Eryth-prog,Eryth-prog,15,GSM5652288_Blood-T-Eff-CD8-Z00000419.hg38_reads,39,99,train
19241505,chr5,140679722,TTAAACCTGAGCCAAGTGAGTGCCAATCCATCCAAAGTCTCAAGAG...,2222222222222222222222222222222222222222222222...,140679820,6,chr5:140678758-140679821,140678758,140679821,Eryth-prog,Eryth-prog,15,GSM5652288_Blood-T-Eff-CD8-Z00000419.hg38_reads,39,99,train
19241506,chr5,140679720,TCTTAAACCTGAGCCAAGTGAGTGCCAATCCATCCAAAGTCTCAAG...,2222222222222222222222222222222222222222222222...,140679819,6,chr5:140678758-140679821,140678758,140679821,Eryth-prog,Eryth-prog,15,GSM5652288_Blood-T-Eff-CD8-Z00000419.hg38_reads,39,99,train
19241507,chr5,140679751,ATCCAAAGTCTCAAGAGCCCAAGTTTAGAAAGATACAGTGAGGTCA...,2222222222222222222222222222222222222222222222...,140679820,6,chr5:140678758-140679821,140678758,140679821,Eryth-prog,Eryth-prog,15,GSM5652288_Blood-T-Eff-CD8-Z00000419.hg38_reads,39,99,train


In [15]:
labels_dict_path = "/home/luna.kuleuven.be/u0169940/Repos/syto/App/labels_dict.json"
with open(labels_dict_path, "r", encoding="utf-8") as f:
    labels_dict = json.load(f)

In [11]:
df.rename(columns={"methylation_ids":"pattern"}, inplace=True)

In [16]:
CpGBetaCountsMethylationAtlas.from_reads(df, "U25.l4.hg19","hg19",labels_dict=labels_dict, output_path="U25.l1.hg38.BetaCountsMethylAtlas.csv")

Aggregating groups: 100%|██████████| 37146/37146 [01:33<00:00, 395.81group/s]
